# Packing puzzles, part 1: the solver

A **packing puzzle** hands you a set of pieces and a board, and asks you to fill the board with them: every piece used
once, every cell covered once, nothing overlapping. A puzzle usually fixes a few pieces in place to start from.

This repository solves such puzzles in general, for any set of pieces on a flat board of any shape or on a 3D one like a
pyramid, and it finds *every* solution, not just one. This notebook shows how: how a game is represented, how
solving it becomes an **exact cover** problem, and how the search works through that problem.

The examples use the 12 pieces and the 11 x 5 board of *IQpuzzler*, a commercial packing puzzle. `2_findings.ipynb`
then puts the solver to use on the puzzle booklets of that game and of its sequel.

Everything here is computed when you run it, in a few seconds.

*Some outputs are long. VS Code cuts cell outputs at 30 lines by default; to see them in full, raise
`notebook.output.textLineLimit` (to 200, say) in your settings.
GitHub and JupyterLab show outputs in full.*

In [1]:
from pugpuzzler.constants import GAMES_DIR

from pugpuzzler.serialization import load_game, load_puzzle
from pugpuzzler.solving import Verbosity, solve_puzzle

from solver_trace import describe, render_block_counts, render_branches, render_cell_counts, walk  # next to this notebook

game = load_game(GAMES_DIR / "IQpuzzler")

## 1. The game

A game is a set of blocks and one or more boards. Printing an empty puzzle shows the board, with every block still to
be placed drawn underneath it. Here, that is the empty 55-cell board and all twelve blocks:

In [2]:
empty = game.puzzles["empty_main"]
print(empty)

Puzzle empty_main

 ┏━━━━━━━━━━━━━━━━━━━━━━┓ 
 ┃                      ┃ 
 ┃                      ┃ 
 ┃                      ┃ 
 ┃                      ┃ 
 ┃                      ┃ 
 ┗━━━━━━━━━━━━━━━━━━━━━━┛ 

A A   B B   C C   D     E     F     G G G   H       I I   J   K K     L   
  A   B B   C     D D   E     F F   G       H H       I   J   K K   L L L 
  A     B   C     D     E E         G         H H   I I   J           L   
            C     D       E                               J               


Before any solving, the `Setup` of a board works out every **placement**: every way one block can lie on the board
(every rotation and mirror image, at every position where it fits). The symmetric blocks have fewer: the square `K`
looks the same in every orientation, so it only has 40 placements, against the 264 of e.g. `A`.

In [3]:
setup = empty.setup
print(f"{'block':<7}{'cells':>5}{'placements':>12}")
for idx, block in setup.blocks.items():
    print(f"{block.letter:<7}{block.count:>5}{len(setup.placement_cells[idx]):>12}")
print(f"{'total':<7}{sum(b.count for b in setup.blocks.values()):>5}{sum(map(len, setup.placement_cells)):>12}")

block  cells  placements
A          4         264
B          5         264
C          5         208
D          5         208
E          5         208
F          3         160
G          5         108
H          5         108
I          5         132
J          4          62
K          4          40
L          5          27
total     55        1789


## 2. Packing as exact cover

Solving a puzzle means choosing one placement per block so that every one of the 55 cells is covered exactly once.
Write the placements as the rows of a table with one column per cell and one column per block, and put a 1 in a row
wherever that placement covers the cell or *is* that block. A solution is then a set of rows with exactly one 1 in
every column: an **exact cover**. The blocks' cells add up to exactly 55, so covering every cell *is* placing every
block, and the other way around.

Nothing in this formulation is specific to this game: a different board or different pieces only change the table.

## 3. The search

The solver is a depth-first search. At each step (a **node**) it picks one decision that has to be made, and branches
on every way of making it. It either branches on

- an **empty cell**, which must be covered by exactly one placement, so it tries every placement that still fits over
  it; or
- an **unplaced block**, which must go somewhere, so it tries every placement of it that still fits.

In the table's terms, both are a column, and the branches are the rows that still have a 1 in it. Every solution makes
that decision exactly one way, so the search finds every solution, and each one exactly once.

The solver counts the options for *every* empty cell and *every* unplaced block, and branches on whichever has the
**fewest**. Choosing the most constrained decision cuts off dead ends the fastest: a count of 1 is a forced move, and a
count of 0 is a dead end, which sends the search straight back up.

This is Knuth's **Algorithm X**, from his 2000 paper [Dancing Links](https://arxiv.org/abs/cs/0011047): pick the
column with the fewest rows left, try each of those rows in turn, and remove everything that conflicts with it. One of
the paper's examples is a packing puzzle too, the twelve pentominoes on a 6 x 10 board, with a column for every piece
and every cell. The paper's title is about how Knuth *implements* Algorithm X: with doubly linked lists, from which
conflicting rows are unlinked and later relinked in place ("dancing links", or DLX). This solver runs the same
algorithm on bitmasks instead; section 5 has the details.

## 4. A worked example

As an example we use a puzzle made up for this tutorial (`demo_puzzle.json`, next to this notebook). Only three pieces
are left to place, so every count below fits in a single digit, and yet in four nodes the search shows everything it
does: it branches on a block, then on a cell, runs into a dead end, backs up, and finds the solution:

In [4]:
puzzle = load_puzzle("demo_puzzle.json", game)
print(puzzle)

Puzzle demo_puzzle

 ┏━━━━━━━━━━━━━━━━━━━━━━┓ 
 ┃K K D D D D J J J J   ┃ 
 ┃K K B B D H H         ┃ 
 ┃G B B B H H     L I I ┃ 
 ┃G E E E H     L L L I ┃ 
 ┃G G G E E       L I I ┃ 
 ┗━━━━━━━━━━━━━━━━━━━━━━┛ 

A A   C C   F   
  A   C     F F 
  A   C         
      C         


`solver_trace` replays the search node by node, using the solver's own functions to count and to choose, so it shows
exactly what the real search does: it tries each node's branches in order, and backs up out of every dead end, until it
finds a solution.

At the root, the solver first counts. For every empty cell on the board, it counts the placements of blocks that still fit over it. Similarly, for every
block it still has to place, it counts the placements of that block that still fit on the board:

In [5]:
nodes, solved = walk(puzzle)   # every node the search visits, up to its first solution
depth, node, step = nodes[0]

print("Number of placements covering each uncovered cell")
print(render_cell_counts(node, step))
print("Number of placements per block")
print(render_block_counts(node, step))

Number of placements covering each uncovered cell
 ┏━━━━━━━━━━━━━━━━━━━━━━┓ 
 ┃                    3 ┃ 
 ┃              5 5 5 4 ┃ 
 ┃            6 6       ┃ 
 ┃          5 9         ┃ 
 ┃          5 8 3       ┃ 
 ┗━━━━━━━━━━━━━━━━━━━━━━┛ 
Number of placements per block
A A             C C             F 
  A             C               F F 
  A             C 
                C 

6 placements    2 placements    10 placements


Every empty cell has at least 3 placements over it, but `C` fits in only 2 places. Two is fewer, so the solver
branches on the **block**:

In [6]:
print(render_branches(node, step))

Branching on block C: 2 placements

branch 1                      branch 2

 ┏━━━━━━━━━━━━━━━━━━━━━━┓      ┏━━━━━━━━━━━━━━━━━━━━━━┓
 ┃K K D D D D J J J J C ┃      ┃K K D D D D J J J J   ┃
 ┃K K B B D H H C C C C ┃      ┃K K B B D H H C C C C ┃
 ┃G B B B H H     L I I ┃      ┃G B B B H H   C L I I ┃
 ┃G E E E H     L L L I ┃      ┃G E E E H     L L L I ┃
 ┃G G G E E       L I I ┃      ┃G G G E E       L I I ┃
 ┗━━━━━━━━━━━━━━━━━━━━━━┛      ┗━━━━━━━━━━━━━━━━━━━━━━┛


The search tries branch 1 first, and counts again:

In [7]:
depth, node, step = nodes[1]

print("Number of placements covering each uncovered cell")
print(render_cell_counts(node, step))
print("Number of placements per block")
print(render_block_counts(node, step))

Number of placements covering each uncovered cell
 ┏━━━━━━━━━━━━━━━━━━━━━━┓ 
 ┃                      ┃ 
 ┃                      ┃ 
 ┃            5 2       ┃ 
 ┃          5 9         ┃ 
 ┃          5 8 3       ┃ 
 ┗━━━━━━━━━━━━━━━━━━━━━━┛ 
Number of placements per block
A A             F 
  A             F F 
  A 

4 placements    7 placements


Now `A` fits in 4 places and `F` in 7, but the empty cell next to `L` in the middle row is covered by only 2, so
this time the solver branches on the **cell**:

In [8]:
print(render_branches(node, step))

Branching on cell (7, 2): 2 placements cover it

branch 1                      branch 2

 ┏━━━━━━━━━━━━━━━━━━━━━━┓      ┏━━━━━━━━━━━━━━━━━━━━━━┓
 ┃K K D D D D J J J J C ┃      ┃K K D D D D J J J J C ┃
 ┃K K B B D H H C C C C ┃      ┃K K B B D H H C C C C ┃
 ┃G B B B H H A A L I I ┃      ┃G B B B H H F F L I I ┃
 ┃G E E E H   A L L L I ┃      ┃G E E E H   F L L L I ┃
 ┃G G G E E   A   L I I ┃      ┃G G G E E       L I I ┃
 ┗━━━━━━━━━━━━━━━━━━━━━━┛      ┗━━━━━━━━━━━━━━━━━━━━━━┛


Again it tries branch 1 first, placing `A`. That leaves three cells that are cut off from each other, and no
placement of the last block, `F`, covers any of them. Every count is 0: a **dead end**.

In [9]:
depth, node, step = nodes[2]

print("Number of placements covering each uncovered cell")
print(render_cell_counts(node, step))
print("Number of placements per block")
print(render_block_counts(node, step))

Number of placements covering each uncovered cell
 ┏━━━━━━━━━━━━━━━━━━━━━━┓ 
 ┃                      ┃ 
 ┃                      ┃ 
 ┃                      ┃ 
 ┃          0           ┃ 
 ┃          0   0       ┃ 
 ┗━━━━━━━━━━━━━━━━━━━━━━┛ 
Number of placements per block
F 
F F 

0 placements


So the search backs up and tries branch 2, placing `F`. The three cells left over are exactly the shape of `A`,
which fits there in one way only: a **forced** move.

In [10]:
depth, node, step = nodes[3]

print("Number of placements covering each uncovered cell")
print(render_cell_counts(node, step))
print("Number of placements per block")
print(render_block_counts(node, step))

Number of placements covering each uncovered cell
 ┏━━━━━━━━━━━━━━━━━━━━━━┓ 
 ┃                      ┃ 
 ┃                      ┃ 
 ┃                      ┃ 
 ┃          1           ┃ 
 ┃          1 1 1       ┃ 
 ┗━━━━━━━━━━━━━━━━━━━━━━┛ 
Number of placements per block
A A 
  A 
  A 

1 placement


Placing it fills the board, and that is the first solution:

In [11]:
print(solved)

Puzzle demo_puzzle

 ┏━━━━━━━━━━━━━━━━━━━━━━┓ 
 ┃K K D D D D J J J J C ┃ 
 ┃K K B B D H H C C C C ┃ 
 ┃G B B B H H F F L I I ┃ 
 ┃G E E E H A F L L L I ┃ 
 ┃G G G E E A A A L I I ┃ 
 ┗━━━━━━━━━━━━━━━━━━━━━━┛ 


Choosing from both kinds is what makes this work. A cell-only search would have started with 3 options at the root
instead of `C`'s 2, and a block-only search would have had 4 (`A`) at the second node instead of the cell's 2.

Here is the whole search up to the first solution, indented by depth:

In [12]:
print(describe(nodes, solved))

block C, 2 branches
  cell (7, 2), 2 branches
    cell (5, 3)
        dead end
    cell (5, 3)
        forced --> *solution!*
  ...


The real solver doesn't stop at the first solution: it keeps going to find every one. Here it backs up to the
root's branch 2 and finds nothing there, so this puzzle's solution is unique:

In [13]:
solution, stats = solve_puzzle(puzzle, verbose=Verbosity.SHOW_SOLUTIONS)

Puzzle demo_puzzle (solution 1)

 ┏━━━━━━━━━━━━━━━━━━━━━━┓ 
 ┃K K D D D D J J J J C ┃ 
 ┃K K B B D H H C C C C ┃ 
 ┃G B B B H H F F L I I ┃ 
 ┃G E E E H A F L L L I ┃ 
 ┃G G G E E A A A L I I ┃ 
 ┗━━━━━━━━━━━━━━━━━━━━━━┛ 
Found 1 solution in 0.00s


## 5. Under the hood

Dancing links keep each column's count up to date as rows are removed and restored. This solver removes nothing: the
board's occupancy is a bitmask with one bit per cell (a 55-cell board fits in one 64-bit word), and so is every
placement. "Does this placement still fit" is then a single AND, placing it or taking it back a single XOR, and each
node simply recounts, stopping every count as soon as it exceeds the smallest found so far. The search keeps its own
explicit stack and is compiled with numba.

`docs/SOLVER_WALKTHROUGH.md` goes through the code, and `docs/SOLVER_NOTES.md` has the measurements behind each design choice,
including the ideas that were tried and removed.

Next, `2_findings.ipynb` uses this solver on two commercial puzzle games, *IQpuzzler* and its sequel *IQpuzzlerPRO*:
it finds every solution of every puzzle in their booklets, and asks how few or how many solutions a puzzle could
possibly have.